# Machine Learning Capstone Project

This notebook runs the full machine learning pipeline: data loading, preprocessing, three models, and evaluation.

| Task | Model | Dataset |
|---|---|---|
| Regression | Linear Regression | California Housing |
| Classification | Logistic Regression | EmpatheticDialogues (Hugging Face) |
| Clustering | K-Means | California Housing |

**How to run:** choose *Runtime > Run all*. The data is downloaded automatically by the cell below.

In [ ]:
!pip install -q datasets

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    silhouette_score,
)

In [ ]:
RANDOM_STATE = 42
SHOW = True
REG_TARGET = "median_house_value"

POSITIVE = ["excited", "proud", "grateful", "impressed", "confident", "hopeful",
            "joyful", "prepared", "content", "caring", "trusting", "faithful"]
NEGATIVE = ["annoyed", "angry", "sad", "lonely", "afraid", "disgusted", "terrified",
            "anxious", "disappointed", "guilty", "furious", "jealous", "embarrassed",
            "devastated", "ashamed", "apprehensive"]

log_lines = []

BASE = ".." if os.path.isdir("../data") else "."
DATA = f"{BASE}/data"
RES = f"{BASE}/results"
os.makedirs(RES, exist_ok=True)

### Get the data
The housing file comes from the GitHub repository and the EmpatheticDialogues file from Hugging Face. This cell only runs when the files are not already in the `data/` folder.

**Before running on Colab:** replace `YOUR_GITHUB_USERNAME` with your GitHub username.

In [ ]:
os.makedirs(DATA, exist_ok=True)

GITHUB_USER = "YOUR_GITHUB_USERNAME"
if not os.path.exists(f"{DATA}/housing.csv"):
    import urllib.request
    url = f"https://raw.githubusercontent.com/{GITHUB_USER}/ML-Capstone-Project/main/data/housing.csv"
    urllib.request.urlretrieve(url, f"{DATA}/housing.csv")

if not os.path.exists(f"{DATA}/empathetic.csv"):
    from datasets import load_dataset
    ds = load_dataset("Estwld/empathetic_dialogues_llm")
    df_emp = pd.concat([split.to_pandas() for split in ds.values()])
    df_emp["first_message"] = df_emp["conversations"].apply(lambda c: c[0]["content"])
    df_emp[["emotion", "situation", "first_message"]].to_csv(f"{DATA}/empathetic.csv", index=False)

### Helper functions
Small functions used many times: logging numbers to a file, saving figures, and cleaning a table.

In [ ]:
def log(text=""):
    print(text)
    log_lines.append(str(text))


def finish(filename):
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    if SHOW:
        plt.show()
    plt.close()


def clean(df):
    print("\nMissing values per column:")
    print(df.isnull().sum())
    for col in df.columns:
        if df[col].isnull().any():
            if pd.api.types.is_numeric_dtype(df[col]):
                df[col] = df[col].fillna(df[col].median())
            else:
                df[col] = df[col].fillna(df[col].mode()[0])
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        df = pd.get_dummies(df, columns=text_cols, drop_first=True)
    return df

## Part A: Regression with Linear Regression (California Housing)

### 1. Data loading and initial inspection
Size, column types, and the target variable (`median_house_value`).

In [ ]:
housing = pd.read_csv(f"{DATA}/housing.csv")
log("===== HOUSING DATASET =====")
log(f"Shape: {housing.shape}")
print("\nColumn data types:")
print(housing.dtypes)
print(housing.head())
log(f"Regression target: {REG_TARGET}")

### 2. Data cleaning and preparation
Missing values are filled with the median, the text column is One-Hot Encoded, and box plots are used to look for outliers.

In [ ]:
housing = clean(housing)

num_cols = housing.select_dtypes(include=np.number).columns.drop(REG_TARGET)
plt.figure(figsize=(14, 6))
sns.boxplot(data=housing[num_cols])
plt.xticks(rotation=45)
plt.ylabel("Value")
plt.title("Box plots of the numerical features (dots are outliers)")
finish(f"{RES}/housing_boxplots.png")

**Outlier decision:** many districts have the price capped at 500001, so the real price is unknown. These rows are removed.

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(housing[REG_TARGET], bins=50)
plt.xlabel("Median house value (USD)")
plt.ylabel("Number of districts")
plt.title("Distribution of house prices (note the spike at the maximum, 500001)")
finish(f"{RES}/housing_target_distribution.png")

max_price = housing[REG_TARGET].max()
n_before = len(housing)
housing = housing[housing[REG_TARGET] < max_price].reset_index(drop=True)
log(f"Removed {n_before - len(housing)} districts with a capped price ({max_price:.0f}); "
    f"{len(housing)} remain")

### 3. Exploratory data analysis
Correlations between the features, and a map of the prices.

In [ ]:
corr = housing.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
plt.figure(figsize=(12, 9))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation between features (red = move together, blue = move opposite)")
finish(f"{RES}/housing_correlation.png")

In [ ]:
plt.figure(figsize=(8, 6))
sc = plt.scatter(housing["longitude"], housing["latitude"],
                 c=housing[REG_TARGET], cmap="plasma", s=6)
plt.colorbar(sc, label="Median house value (USD)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("House prices across California (each dot is one district)")
finish(f"{RES}/housing_price_map.png")

### 4. Scaling and splitting
80% train, 20% test, then `StandardScaler` fitted on the training set only.

In [ ]:
X_reg = housing.drop(columns=[REG_TARGET])
y_reg = housing[REG_TARGET]
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE
)
scaler_reg = StandardScaler()
Xr_train = scaler_reg.fit_transform(Xr_train)
Xr_test = scaler_reg.transform(Xr_test)

### 5. Linear Regression
Train the model, predict on the test set, and calculate MSE, RMSE and R2.

In [ ]:
log("\n===== Linear Regression =====")
lin_reg = LinearRegression()
lin_reg.fit(Xr_train, yr_train)
yr_pred = lin_reg.predict(Xr_test)

mse = mean_squared_error(yr_test, yr_pred)
rmse = np.sqrt(mse)
r2 = r2_score(yr_test, yr_pred)
log(f"MSE:  {mse:.4f}")
log(f"RMSE: {rmse:.4f}")
log(f"R2:   {r2:.4f}")

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(yr_test, yr_pred, s=5, alpha=0.3)
lims = [min(yr_test.min(), yr_pred.min()), max(yr_test.max(), yr_pred.max())]
plt.plot(lims, lims, color="red", label="Perfect prediction")
plt.xlabel("Actual price (USD)")
plt.ylabel("Predicted price (USD)")
plt.title("Predicted vs actual house prices")
plt.legend()
finish(f"{RES}/regression_predicted_vs_actual.png")

In [ ]:
coef = pd.Series(lin_reg.coef_, index=X_reg.columns).sort_values()
plt.figure(figsize=(8, 6))
plt.barh(coef.index, coef.values, color=["tab:red" if v < 0 else "tab:green" for v in coef.values])
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Effect on price (green = raises it, red = lowers it)")
plt.title("What drives house prices in the model")
finish(f"{RES}/regression_feature_effects.png")

## Part B: Classification with Logistic Regression (EmpatheticDialogues)

The text of a situation is used to predict whether the emotion is positive (1) or negative (0).

### 1. Data loading and initial inspection

In [ ]:
emp = pd.read_csv(f"{DATA}/empathetic.csv")
log("\n\n===== EMPATHETIC DIALOGUES DATASET =====")
log(f"Shape: {emp.shape}")
print("\nColumn data types:")
print(emp.dtypes)
print(emp.head())

print("\nMissing values per column:")
print(emp.isnull().sum())

### 2. Data cleaning and preparation
The 32 emotions are grouped into positive and negative to build a binary target. Four ambiguous emotions are left out.

In [ ]:
emp = emp.dropna(subset=["situation", "emotion"])
emp_all = emp.copy()

counts = emp_all["emotion"].value_counts()
bar_colors = ["tab:green" if e in POSITIVE else "tab:red" if e in NEGATIVE else "tab:gray"
              for e in counts.index]
plt.figure(figsize=(8, 9))
plt.barh(counts.index[::-1], counts.values[::-1], color=bar_colors[::-1])
plt.xlabel("Number of conversations")
plt.title("The 32 emotions and how they were grouped")
plt.legend(handles=[Patch(color="tab:green", label="Positive"),
                    Patch(color="tab:red", label="Negative"),
                    Patch(color="tab:gray", label="Left out (ambiguous)")])
finish(f"{RES}/empathetic_emotion_groups.png")

In [ ]:
emp = emp[emp["emotion"].isin(POSITIVE + NEGATIVE)].copy()
emp["label"] = emp["emotion"].isin(POSITIVE).astype(int)
CLF_TARGET = "label"
log(f"Classification target: {CLF_TARGET} (1 = positive, 0 = negative)")
log(emp[CLF_TARGET].value_counts().to_string())

### 3. Exploratory data analysis

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(emp["situation"].str.split().str.len(), bins=40)
plt.xlabel("Words per situation")
plt.ylabel("Number of conversations")
plt.title("Length of the situation texts")
finish(f"{RES}/empathetic_text_length.png")

### 4. Splitting and text vectorization
`TfidfVectorizer` turns the text into numbers.

In [ ]:
Xc_text_train, Xc_text_test, yc_train, yc_test = train_test_split(
    emp["situation"], emp[CLF_TARGET], test_size=0.2, random_state=RANDOM_STATE,
    stratify=emp[CLF_TARGET]
)
tfidf = TfidfVectorizer(max_features=5000, stop_words="english")
Xc_train = tfidf.fit_transform(Xc_text_train)
Xc_test = tfidf.transform(Xc_text_test)

### 5. Logistic Regression
Accuracy, Confusion Matrix, Precision and Recall.

In [ ]:
log("\n===== Logistic Regression =====")
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(Xc_train, yc_train)
yc_pred = log_reg.predict(Xc_test)

acc = accuracy_score(yc_test, yc_pred)
cm = confusion_matrix(yc_test, yc_pred)
tn, fp, fn, tp = cm.ravel()
prec = precision_score(yc_test, yc_pred)
rec = recall_score(yc_test, yc_pred)

log(f"Accuracy:  {acc:.4f}")
log(f"Confusion Matrix:\n{cm}")
log(f"TP={tp}, FP={fp}, TN={tn}, FN={fn}")
log(f"Precision: {prec:.4f}")
log(f"Recall:    {rec:.4f}")

In [ ]:
cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100
annot = np.array([[f"{cm[i, j]}\n{cm_pct[i, j]:.1f}%" for j in range(2)] for i in range(2)])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=annot, fmt="", cmap="Blues",
            xticklabels=["Negative", "Positive"], yticklabels=["Negative", "Positive"])
plt.xlabel("Predicted emotion")
plt.ylabel("Actual emotion")
plt.title("Confusion Matrix")
finish(f"{RES}/confusion_matrix.png")

In [ ]:
words = tfidf.get_feature_names_out()
coefs = log_reg.coef_[0]
order = np.argsort(coefs)
idx = np.concatenate([order[:10], order[-10:]])
plt.figure(figsize=(7, 7))
plt.barh(words[idx], coefs[idx], color=["tab:red"] * 10 + ["tab:green"] * 10)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Weight in the model")
plt.title("Words the model uses to tell negative from positive")
finish(f"{RES}/classification_top_words.png")

## Part C: Clustering with K-Means (California Housing, without the label)

### 1. Choosing K with the Elbow Method

In [ ]:
log("\n===== K-Means Clustering =====")
X_cluster = housing.drop(columns=[REG_TARGET])
X_cluster_scaled = StandardScaler().fit_transform(X_cluster)

k_values = list(range(1, 11))
inertias = []
for k in k_values:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_cluster_scaled)
    inertias.append(km.inertia_)

In [ ]:
p1 = np.array([k_values[0], inertias[0]])
p2 = np.array([k_values[-1], inertias[-1]])
distances = []
for k, inertia in zip(k_values, inertias):
    p = np.array([k, inertia])
    d = p2 - p1
    v = p1 - p
    distances.append(abs(d[0] * v[1] - d[1] * v[0]) / np.linalg.norm(d))
optimal_k = k_values[int(np.argmax(distances))]
log(f"Optimal K (Elbow): {optimal_k}")

plt.figure(figsize=(6, 4))
plt.plot(k_values, inertias, marker="o")
plt.axvline(optimal_k, color="red", linestyle="--", label=f"Chosen K = {optimal_k}")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia (lower = tighter clusters)")
plt.title("Elbow Method")
plt.legend()
finish(f"{RES}/elbow_plot.png")

### 2. Fit the final model and calculate the Silhouette Score

In [ ]:
kmeans = KMeans(n_clusters=optimal_k, random_state=RANDOM_STATE, n_init=10)
cluster_labels = kmeans.fit_predict(X_cluster_scaled)

sil = silhouette_score(X_cluster_scaled, cluster_labels)
log(f"Silhouette Score: {sil:.4f}")

### 3. Visualize and describe the clusters

In [ ]:
plt.figure(figsize=(8, 6))
for c in range(optimal_k):
    members = cluster_labels == c
    plt.scatter(X_cluster.loc[members, "longitude"], X_cluster.loc[members, "latitude"],
                s=5, label=f"Cluster {c}")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title(f"K-Means clusters on the map (K = {optimal_k})")
plt.legend(markerscale=3)
finish(f"{RES}/clusters.png")

In [ ]:
cluster_summary = (
    housing.assign(cluster=cluster_labels)
    .groupby("cluster")[["median_income", REG_TARGET, "housing_median_age", "population"]]
    .mean()
    .round(1)
)
cluster_summary["districts"] = pd.Series(cluster_labels).value_counts().sort_index()
cluster_summary.to_csv(f"{RES}/cluster_summary.csv")
log("\nCluster summary (average per cluster):")
log(cluster_summary.to_string())

## Save all numbers
Everything printed above is saved in `results/metrics.txt` for the report.

In [ ]:
with open(f"{RES}/metrics.txt", "w") as f:
    f.write("\n".join(log_lines))
print("\nAll numbers were saved to results/metrics.txt")